In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("F1_Preguntas")
    .enableHiveSupport()
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

spark.sql("USE f1_dw")

# Pregunta 5
q5 = spark.sql("""
    WITH dnf_por_escuderia AS (
        SELECT
            r.constructorId,
            SUM(CASE WHEN r.positionText = 'R' THEN 1 ELSE 0 END)          AS dnf_count,
            SUM(CASE WHEN r.positionText <> 'W' THEN 1 ELSE 0 END)         AS starts,
            ROUND(
                SUM(CASE WHEN r.positionText = 'R' THEN 1 ELSE 0 END) * 100.0
                / SUM(CASE WHEN r.positionText <> 'W' THEN 1 ELSE 0 END), 2
            )                                                              AS dnf_rate_pct
        FROM f1_dw.fact_results r
        JOIN f1_dw.dim_race ra ON r.raceId = ra.raceId
        WHERE ra.season BETWEEN 2016 AND 2025
        GROUP BY r.constructorId
    ),
    posicion_final AS (
        SELECT
            constructorId,
            COUNT(*)                    AS seasons,
            ROUND(AVG(position), 2)     AS avg_final_position
        FROM f1_dw.fact_constructor_standings
        WHERE season BETWEEN 2016 AND 2025
        GROUP BY constructorId
    )
    SELECT
        c.name                AS constructor,
        d.dnf_count,
        d.starts,
        d.dnf_rate_pct,
        p.seasons,
        p.avg_final_position
    FROM dnf_por_escuderia d
    JOIN posicion_final p ON d.constructorId = p.constructorId
    JOIN f1_dw.dim_constructor c ON d.constructorId = c.constructorId
    ORDER BY d.dnf_count DESC
""")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-07-11T19:29:55,005 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-11T19:29:56,236 WARN [Thread-4] org.apache.spark.util.Utils - Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2026-07-11T19:29:56,237 WARN [Thread-4] org.apache.spark.util.Utils - Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
2026-07-11T19:30:02,911 INFO [Thread-4] org.apache.hadoop.hive.conf.HiveConf - Found configuration file file:/home/ort/spark/conf/hive-site.xml
2026-07-11T19:30:03,185 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.metastore.wm.default.pool.size does not exist
2026-07-11T19:30:03,185 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.llap.task.scheduler.preempt.independent does not exist
2026-07-11T19:30:03,185 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of 

In [3]:
from pyspark.sql.functions import col

(q5
    .withColumn("dnf_rate_pct", col("dnf_rate_pct").cast("double"))
    .write.mode("overwrite").parquet("ort/ANL/pregunta5_dnf_escuderias"))

In [4]:
spark.read.parquet("ort/ANL/pregunta5_dnf_escuderias").printSchema()

root
 |-- constructor: string (nullable = true)
 |-- dnf_count: long (nullable = true)
 |-- starts: long (nullable = true)
 |-- dnf_rate_pct: double (nullable = true)
 |-- seasons: long (nullable = true)
 |-- avg_final_position: double (nullable = true)

